# Exercice à faire à la maison — Séance 3 : Analyse et visualisation de réseaux

**Objectif :** cet exercice a pour but de vous familiariser davantage avec l'analyse de réseaux en Python, en remobilisant les méthodes vues pendant la Séance 3 (réseau biparti, projection, centralité, détection de communautés) sur un **nouveau** jeu de données.

## Le jeu de données

Le jeu de données proposé est une liste d'individus avec les associations auxquelles ils appartiennent (clubs étudiants, sociétés savantes, associations professionnelles, fraternités...). Il s'agit donc, comme pour le réseau université–employeur vu en cours, d'un **réseau d'affiliation de type biparti**. L'enjeu est d'identifier d'éventuels *patterns* d'affiliation, tels que :

- des individus partageant les mêmes organisations ;
- des combinaisons d'organisations plus fréquentes que d'autres ;
- des sous-groupes, ou communautés, fondés sur ces affiliations.

<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 Gardez à l'esprit que l'affiliation ne constitue qu'un lien <b>indirect</b> : appartenir à la même association n'implique pas nécessairement d'interaction sociale réelle entre deux personnes. On fait ici l'hypothèse, raisonnable mais non démontrée, que des individus membres des mêmes organisations ont davantage de chances de s'être rencontrés — ou, à défaut, de partager des centres d'intérêt communs.
</div>

Les données sont tirées du même annuaire d'étudiants chinois aux États-Unis paru en 1917 (*Who's Who of American Returned Students*) déjà utilisé pour l'exercice de la Séance 1 (`youmei.csv`). Trois fichiers sont fournis :

| Fichier | Contenu |
|---|---|
| `association.csv` | Pour chaque étudiant, les associations, clubs, sociétés et fraternités auxquels il appartient (585 lignes). C'est le fichier principal de cet exercice. |
| `youmei.csv` | Les données biographiques des étudiants (genre, discipline, province de naissance...), vues en Séance 1 — utile ici pour **enrichir** le réseau avec des attributs. |
| `affiliation.csv` | Un fichier plus large, incluant toutes les affiliations institutionnelles des étudiants (écoles, employeurs, administrations, associations...) — dont `association.csv` est en réalité un sous-ensemble filtré. Il servira pour la section « Pour aller plus loin ». |

Les trois fichiers partagent une colonne `DocId` commune, qui permet de les relier entre eux — exactement comme on relierait plusieurs tables dans une base de données relationnelle.

<div class="alert alert-danger" role="alert" style="background-color:#f8d7da;padding:10px;border-radius:5px;">
📁 <b>Données</b> : placez <code>association.csv</code>, <code>youmei.csv</code> et <code>affiliation.csv</code> dans un dossier <code>data/</code> à côté de ce notebook.
</div>

<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 <b>Comment utiliser ce notebook</b> : chaque question est suivie d'une cellule de code vide, à compléter. La solution est ensuite donnée dans un bloc repliable — essayez toujours de résoudre la question par vous-même avant de la consulter.
</div>


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from networkx.algorithms import bipartite

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

print("Bibliothèques chargées avec succès ✅")


## 1. Charger les données

**Question 1.** Chargez le fichier `association.csv` dans un dataframe nommé `association`, et inspectez-le (dimensions, colonnes, types, valeurs manquantes).


In [ ]:
# ✏️ Votre code ici




<details>
<summary>▶️ Voir la solution</summary>

```python
association = pd.read_csv("data/association.csv")

association.shape
association.columns
association.dtypes
association.isna().sum()
association.head()
```

Le fichier contient 6 colonnes : `DocId` (identifiant de l'étudiant), `name` (nom en caractères chinois), `Organization` (nom de l'association), `Position`, `category` et `category_main` (toutes deux égales à `"association"` ou une sous-catégorie).
</details>


**Question 2.** Créez la liste de liens (*edge list*) simplifiée : pour chaque paire unique (étudiant, association), une seule ligne. Combien de liens uniques obtenez-vous ?

<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 Un même étudiant peut apparaître plusieurs fois pour la même association dans le fichier source (doublons) — pensez à <code>.drop_duplicates()</code>, vu en Séance 1.
</div>


In [ ]:
# ✏️ Votre code ici




<details>
<summary>▶️ Voir la solution</summary>

```python
links = association[["DocId", "Organization"]].drop_duplicates()

print(f"{len(links)} liens uniques étudiant-association")
links.head()
```
</details>


**Question 3.** Créez la liste de nœuds, en distinguant clairement les étudiants et les organisations (par exemple, avec une colonne `node_type`).

<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 Les étudiants sont dans la colonne <code>DocId</code>, les organisations dans la colonne <code>Organization</code> — il s'agit ici de lister les valeurs uniques de chaque côté, comme pour <code>universities</code> et <code>employers</code> en cours.
</div>


In [ ]:
# ✏️ Votre code ici




<details>
<summary>▶️ Voir la solution</summary>

```python
students = links["DocId"].unique()
organizations = links["Organization"].unique()

print(f"{len(students)} étudiants, {len(organizations)} associations")

nodes = pd.concat([
    pd.DataFrame({"node": students, "node_type": "Student"}),
    pd.DataFrame({"node": organizations, "node_type": "Association"})
], ignore_index=True)

nodes.head()
```
</details>


## 2. Réseau biparti

**Question 4.** À partir de la liste de liens, construisez le réseau biparti étudiants–associations avec NetworkX, en marquant chaque nœud avec son type (`node_type`, comme en cours).


In [ ]:
# ✏️ Votre code ici




<details>
<summary>▶️ Voir la solution</summary>

```python
B = nx.Graph()
B.add_nodes_from(students, bipartite=0, node_type="Student")
B.add_nodes_from(organizations, bipartite=1, node_type="Association")

for _, row in links.iterrows():
    B.add_edge(row["DocId"], row["Organization"])

print(f"{B.number_of_nodes()} nœuds, {B.number_of_edges()} liens")
```
</details>


**Question 5.** Visualisez le réseau biparti en distinguant les étudiants et les associations par une couleur différente.

<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 Le réseau complet risque d'être bien trop dense pour être lisible d'un seul coup d'œil (près de 500 nœuds !). Comme en cours, filtrez d'abord pour ne garder que les nœuds ayant au moins 2 connexions (<code>B.degree()</code>), avant de dessiner.
</div>


In [ ]:
# ✏️ Votre code ici




<details>
<summary>▶️ Voir la solution</summary>

```python
degrees = dict(B.degree())
keep = [n for n, d in degrees.items() if d >= 2]
B_filtered = B.subgraph(keep).copy()

print(f"{B_filtered.number_of_nodes()} nœuds, {B_filtered.number_of_edges()} liens (seuil >= 2)")

fig, ax = plt.subplots(figsize=(12, 10))
pos = nx.spring_layout(B_filtered, seed=42, k=0.4)

node_colors = ["#4C72B0" if B_filtered.nodes[n]["node_type"] == "Student" else "#DD8452" for n in B_filtered.nodes()]

nx.draw_networkx_nodes(B_filtered, pos, node_color=node_colors, node_size=120, ax=ax)
nx.draw_networkx_edges(B_filtered, pos, edge_color="lightgray", ax=ax)
# On n'affiche les étiquettes que pour les associations, pour ne pas surcharger le graphique
org_labels = {n: n for n in B_filtered.nodes() if B_filtered.nodes[n]["node_type"] == "Association"}
nx.draw_networkx_labels(B_filtered, pos, labels=org_labels, font_size=6, ax=ax)

ax.set_title("Réseau biparti étudiants (bleu) – associations (orange)")
ax.axis("off")
plt.show()
```
</details>


**Question 6.** Calculez les centralités de degré des nœuds du réseau biparti. Quels sont les **étudiants** et les **associations** les plus centraux ? Comment interpréter ces résultats ?

<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 Puisque le réseau contient deux types de nœuds très différents (étudiants et associations), calculez et classez les centralités <b>séparément pour chaque type</b> — comparer directement le degré d'un étudiant à celui d'une association n'aurait pas grand sens.
</div>


In [ ]:
# ✏️ Votre code ici




<details>
<summary>▶️ Voir la solution</summary>

```python
degree_centrality = nx.degree_centrality(B)

# On sépare les résultats selon le type de nœud
students_centrality = {n: c for n, c in degree_centrality.items() if B.nodes[n]["node_type"] == "Student"}
orgs_centrality = {n: c for n, c in degree_centrality.items() if B.nodes[n]["node_type"] == "Association"}

top_students = sorted(students_centrality.items(), key=lambda x: -x[1])[:10]
top_orgs = sorted(orgs_centrality.items(), key=lambda x: -x[1])[:10]

print("Étudiants les plus centraux :", top_students)
print("\nAssociations les plus centrales :", top_orgs)
```

**Interprétation** : côté associations, ce sont sans surprise les grandes organisations généralistes (la *Chinese Students' Alliance*, le *Cosmopolitan Club*...) qui dominent — elles fédèrent un grand nombre d'étudiants aux profils variés. Côté étudiants, les individus en tête sont ceux qui cumulent le plus grand nombre d'appartenances associatives : potentiellement des figures particulièrement engagées dans la vie associative, ou dont la notice biographique était simplement plus détaillée que celle des autres (un biais qu'il faut garder à l'esprit, comme toujours avec des données historiques).
</details>


**Question 7.** Enrichissez la visualisation précédente en ajustant la **taille** des nœuds (et éventuellement des étiquettes) selon leur centralité, de façon à mettre en valeur visuellement les étudiants et les associations les plus centraux.


In [ ]:
# ✏️ Votre code ici




<details>
<summary>▶️ Voir la solution</summary>

```python
fig, ax = plt.subplots(figsize=(12, 10))
pos = nx.spring_layout(B_filtered, seed=42, k=0.4)

node_colors = ["#4C72B0" if B_filtered.nodes[n]["node_type"] == "Student" else "#DD8452" for n in B_filtered.nodes()]
node_sizes = [degree_centrality[n] * 3000 + 40 for n in B_filtered.nodes()]

nx.draw_networkx_nodes(B_filtered, pos, node_color=node_colors, node_size=node_sizes, alpha=0.85, ax=ax)
nx.draw_networkx_edges(B_filtered, pos, edge_color="lightgray", ax=ax)

# On n'étiquette que les nœuds les plus centraux, pour ne pas surcharger le graphique
important_nodes = {n: n for n in B_filtered.nodes() if degree_centrality[n] > 0.02}
nx.draw_networkx_labels(B_filtered, pos, labels=important_nodes, font_size=7, ax=ax)

ax.set_title("Réseau biparti — taille des nœuds proportionnelle à la centralité de degré")
ax.axis("off")
plt.show()
```
</details>


## 3. Projections

**Question 8.** Projetez le réseau biparti étudiants–associations en deux réseaux à un seul type de nœuds : un réseau **étudiants–étudiants** (deux étudiants sont reliés s'ils partagent au moins une association) et un réseau **associations–associations** (deux associations sont reliées si elles partagent au moins un membre).


In [ ]:
# ✏️ Votre code ici




<details>
<summary>▶️ Voir la solution</summary>

```python
student_nodes = {n for n, d in B.nodes(data=True) if d["node_type"] == "Student"}
org_nodes = {n for n, d in B.nodes(data=True) if d["node_type"] == "Association"}

G_students = bipartite.weighted_projected_graph(B, student_nodes)
G_orgs = bipartite.weighted_projected_graph(B, org_nodes)

print(f"Réseau étudiants : {G_students.number_of_nodes()} nœuds, {G_students.number_of_edges()} liens")
print(f"Réseau associations : {G_orgs.number_of_nodes()} nœuds, {G_orgs.number_of_edges()} liens")
```
</details>


**Question 9.** Visualisez les deux nouveaux réseaux (en deux graphiques séparés).

<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 Pensez à filtrer sur le poids des liens (<code>weight</code>) si le résultat est trop dense pour être lisible, comme pour <code>G_univ</code> en cours.
</div>


In [ ]:
# ✏️ Votre code ici




<details>
<summary>▶️ Voir la solution</summary>

```python
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

pos1 = nx.spring_layout(G_students, seed=42, k=0.3)
sizes1 = [G_students.degree(n) * 8 for n in G_students.nodes()]
nx.draw_networkx_nodes(G_students, pos1, node_size=sizes1, node_color="#4C72B0", alpha=0.7, ax=axes[0])
nx.draw_networkx_edges(G_students, pos1, edge_color="lightgray", width=0.4, ax=axes[0])
axes[0].set_title("Réseau étudiants–étudiants (co-affiliation)")
axes[0].axis("off")

pos2 = nx.spring_layout(G_orgs, seed=42, k=0.4)
sizes2 = [G_orgs.degree(n) * 8 for n in G_orgs.nodes()]
nx.draw_networkx_nodes(G_orgs, pos2, node_size=sizes2, node_color="#DD8452", alpha=0.7, ax=axes[1])
nx.draw_networkx_edges(G_orgs, pos2, edge_color="lightgray", width=0.4, ax=axes[1])
axes[1].set_title("Réseau associations–associations (membres partagés)")
axes[1].axis("off")

plt.tight_layout()
plt.show()
```
</details>


**Question 10.** Calculez et comparez les mesures globales des deux réseaux : ordre (nombre de nœuds), densité, nombre de composantes connexes, et diamètre (calculé sur la composante principale).

<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 Le <b>diamètre</b> d'un réseau (<code>nx.diameter()</code>) est la plus longue des plus courtes distances entre deux nœuds — une mesure de la « largeur » du réseau. Il ne peut être calculé que sur un réseau <b>connexe</b> (ou une composante connexe) : isolez d'abord la plus grande composante avec <code>max(nx.connected_components(G), key=len)</code>, comme vu en cours.
</div>


In [ ]:
# ✏️ Votre code ici




<details>
<summary>▶️ Voir la solution</summary>

```python
def resume_reseau(G, nom):
    largest_cc = max(nx.connected_components(G), key=len)
    G_main = G.subgraph(largest_cc).copy()
    return {
        "réseau": nom,
        "ordre": G.number_of_nodes(),
        "densité": round(nx.density(G), 4),
        "composantes connexes": nx.number_connected_components(G),
        "diamètre (composante principale)": nx.diameter(G_main)
    }

pd.DataFrame([resume_reseau(G_students, "Étudiants"), resume_reseau(G_orgs, "Associations")])
```

Le réseau des étudiants est nettement plus **dense** que celui des associations : c'est cohérent, puisque chaque association relie souvent plusieurs dizaines d'étudiants entre eux d'un coup (créant beaucoup de liens), alors que chaque étudiant n'appartient en général qu'à une poignée d'associations.
</details>


**Question 11.** Calculez les centralités (degré, intermédiarité, proximité, vecteur propre) pour chacun des deux réseaux projetés. Comparez avec les centralités pondérées obtenues sur le réseau biparti (question 6) : les classements des individus/associations les plus centraux changent-ils ?

<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 Les centralités <code>closeness</code> et <code>eigenvector</code> nécessitent un réseau connexe : calculez-les sur la composante principale de chaque réseau (<code>G_students_main</code>, <code>G_orgs_main</code>), comme en cours.
</div>


In [ ]:
# ✏️ Votre code ici




<details>
<summary>▶️ Voir la solution</summary>

```python
largest_cc_students = max(nx.connected_components(G_students), key=len)
G_students_main = G_students.subgraph(largest_cc_students).copy()

centrality_students = pd.DataFrame({
    "degree": nx.degree_centrality(G_students_main),
    "betweenness": nx.betweenness_centrality(G_students_main, weight="weight"),
    "closeness": nx.closeness_centrality(G_students_main),
    "eigenvector": nx.eigenvector_centrality(G_students_main, weight="weight", max_iter=1000)
})
centrality_students.sort_values("degree", ascending=False).head(10)

largest_cc_orgs = max(nx.connected_components(G_orgs), key=len)
G_orgs_main = G_orgs.subgraph(largest_cc_orgs).copy()

centrality_orgs = pd.DataFrame({
    "degree": nx.degree_centrality(G_orgs_main),
    "betweenness": nx.betweenness_centrality(G_orgs_main, weight="weight"),
    "closeness": nx.closeness_centrality(G_orgs_main),
    "eigenvector": nx.eigenvector_centrality(G_orgs_main, weight="weight", max_iter=1000)
})
centrality_orgs.sort_values("degree", ascending=False).head(10)
```

Sur le réseau biparti (question 6), la centralité de degré reflétait simplement le **nombre d'appartenances** d'un étudiant. Sur le réseau projeté, elle reflète plutôt le nombre d'**autres étudiants** avec qui il partage au moins une association — une notion assez proche, mais pas strictement identique : un étudiant membre d'une seule association, mais très fréquentée, peut ainsi devenir très central dans le réseau projeté sans l'être particulièrement dans le réseau biparti.
</details>


**Question 12.** Recherchez des communautés d'étudiants (dans `G_students`) avec l'algorithme de Louvain, puis avec un second algorithme de votre choix (par exemple `nx.community.greedy_modularity_communities()`). Comparez le nombre de communautés obtenues et leur modularité.


In [ ]:
# ✏️ Votre code ici




<details>
<summary>▶️ Voir la solution</summary>

```python
communities_louvain = nx.community.louvain_communities(G_students, weight="weight", seed=42)
communities_greedy = nx.community.greedy_modularity_communities(G_students, weight="weight")

print(f"Louvain : {len(communities_louvain)} communautés, modularité = {nx.community.modularity(G_students, communities_louvain, weight='weight'):.3f}")
print(f"Greedy modularity : {len(communities_greedy)} communautés, modularité = {nx.community.modularity(G_students, communities_greedy, weight='weight'):.3f}")
```

Les deux algorithmes n'aboutissent pas nécessairement au même découpage ni au même nombre de communautés — c'est une bonne illustration du fait que la détection de communautés reste une **approximation** statistique, sensible à l'algorithme choisi, et non une vérité unique à découvrir.
</details>


## 4. Pour aller plus loin

Ces questions sont plus ouvertes ; elles vous invitent à mobiliser les outils de la Séance 3 dans des directions un peu différentes.

### A. Enrichir le réseau avec les données biographiques (`youmei.csv`)

**Question 13.** Chargez `youmei.csv` et associez à chaque étudiant du réseau son genre et sa discipline (colonnes `gender` et `field_group`), en les reliant par `DocId`. Redessinez le réseau `G_students` en colorant les nœuds selon le genre, puis selon la discipline. Observez-vous des sous-groupes visuellement cohérents avec l'un de ces deux attributs ?

<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 Un dictionnaire <code>{DocId: valeur}</code>, construit avec <code>youmei.set_index("DocId")["gender"].to_dict()</code>, se prête bien à une utilisation avec <code>nx.set_node_attributes()</code>, vu en Séance 3.
</div>


In [ ]:
# ✏️ Votre code ici




<details>
<summary>▶️ Voir une piste de solution</summary>

```python
youmei = pd.read_csv("data/youmei.csv")

gender_map = youmei.set_index("DocId")["gender"].to_dict()
field_map = youmei.set_index("DocId")["field_group"].to_dict()

nx.set_node_attributes(G_students, gender_map, "gender")
nx.set_node_attributes(G_students, field_map, "field")

color_map = {"Male": "#4C72B0", "Female": "#C44E52"}
node_colors = [color_map.get(G_students.nodes[n].get("gender"), "#CCCCCC") for n in G_students.nodes()]

fig, ax = plt.subplots(figsize=(11, 9))
pos = nx.spring_layout(G_students, seed=42, k=0.3)
nx.draw_networkx_nodes(G_students, pos, node_color=node_colors, node_size=60, alpha=0.8, ax=ax)
nx.draw_networkx_edges(G_students, pos, edge_color="lightgray", width=0.3, ax=ax)
ax.set_title("Réseau des étudiants coloré par genre")
ax.axis("off")
plt.show()
```

Avec seulement 18 étudiantes sur 401 au total (voir l'exercice de la Séance 1), ne vous attendez pas à un clivage spectaculaire par genre — mais vous devriez repérer, en comparant avec la coloration par discipline, que certaines associations (en particulier les clubs professionnels ou disciplinaires) rassemblent des profils nettement plus homogènes.
</details>


### B. Généraliser le pipeline à `affiliation.csv`

Le fichier `affiliation.csv` est un fichier plus large : `association.csv` en est en réalité un **sous-ensemble**, filtré sur `category_main == "association"`. `affiliation.csv` contient aussi d'autres types d'affiliations institutionnelles (`education`, `government`, `business`, `administration`...).

**Question 14.** Chargez `affiliation.csv` et vérifiez cette relation d'inclusion (le nombre de lignes avec `category_main == "association"` correspond-il à celui de `association.csv` ?).

**Question 15.** Choisissez une autre valeur de `category_main` (par exemple `"education"`, les écoles fréquentées) et reproduisez tout le pipeline de cet exercice sur ce nouveau sous-ensemble : réseau biparti, projection, centralités, communautés. La structure du réseau obtenu vous semble-t-elle différente de celle du réseau associatif ? Pourquoi, selon vous ?


In [ ]:
# ✏️ Votre code ici — utilisez autant de cellules que nécessaire





<details>
<summary>▶️ Voir une piste de solution</summary>

```python
# Question 14
affiliation = pd.read_csv("data/affiliation.csv", encoding="utf-8-sig")

print(affiliation["category_main"].value_counts())
n_association_in_affiliation = (affiliation["category_main"] == "association").sum()
print(n_association_in_affiliation, "lignes 'association' dans affiliation.csv, contre", len(association), "dans association.csv")

# Question 15 : exemple avec "education"
education = affiliation[affiliation["category_main"] == "education"]
links_edu = education[["DocId", "Organization"]].drop_duplicates()
edges_edu = links_edu.groupby(["DocId", "Organization"]).size().reset_index(name="weight")

B_edu = nx.Graph()
B_edu.add_nodes_from(edges_edu["DocId"].unique(), bipartite=0, node_type="Student")
B_edu.add_nodes_from(edges_edu["Organization"].unique(), bipartite=1, node_type="School")
for _, row in edges_edu.iterrows():
    B_edu.add_edge(row["DocId"], row["Organization"])

print(f"{B_edu.number_of_nodes()} nœuds, {B_edu.number_of_edges()} liens")
print("densité :", nx.density(B_edu))

student_nodes_edu = {n for n, d in B_edu.nodes(data=True) if d["node_type"] == "Student"}
G_schools = bipartite.weighted_projected_graph(B_edu, {n for n, d in B_edu.nodes(data=True) if d["node_type"] == "School"})

communities_edu = nx.community.louvain_communities(G_schools, weight="weight", seed=42)
print(f"{len(communities_edu)} communautés d'écoles")
```

On peut s'attendre à un réseau « éducation » structurellement différent du réseau associatif : les parcours scolaires sont en général plus séquentiels et concentrés (une poignée de grandes universités regroupent l'essentiel des étudiants), alors que les appartenances associatives sont plus dispersées et cumulatives (un même étudiant appartient souvent à plusieurs associations simultanément).
</details>


### C. Au-delà de la communauté : l'équivalence structurale *(pour les plus curieux·ses)*

La détection de communautés (section 3) regroupe des nœuds **directement ou indirectement connectés** entre eux. Il existe une question voisine, mais différente : deux étudiants peuvent-ils occuper une position **structurellement équivalente** dans le réseau — c'est-à-dire avoir un profil d'appartenances très similaire — sans jamais avoir été directement reliés par une association commune ? C'est la logique de l'**équivalence structurale** (et des méthodes de *blockmodeling* associées), un peu plus avancée, que nous n'avons pas eu le temps de couvrir en Séance 3.

Si le sujet vous intéresse, une bonne façon de commencer à l'explorer avec les outils déjà vus aujourd'hui consiste à considérer que des étudiants aux profils d'affiliation très similaires (même sans lien direct) ont de bonnes chances de se retrouver dans la **même communauté** détectée à la question 12 — la détection de communautés que vous connaissez déjà constitue ainsi une première approximation, imparfaite mais accessible, de cette idée d'équivalence. Nous y reviendrons si l'occasion s'en présente dans une séance ultérieure.


---

## Pour conclure

Si vous êtes arrivé·e au bout de cet exercice sans trop de difficulté, les fondamentaux de la Séance 3 sont bien acquis. Si certaines questions vous ont posé plus de problèmes — en particulier la construction du réseau biparti ou la logique de projection — n'hésitez pas à retourner au notebook de cours correspondant avant de continuer.
